In [ ]:
""" 
Script to extract and save source data for Supplementary Figure 5, which includes:
- Bar plot summary data (group means & SEMs) for each age group and phase.
- Individual subject data (long format) for lines/distribution overlay.
- Spatial channel topography averages for the exponent parameter (for both fixed and knee models).
- LME results for both fixed and knee models, combined into a single file.
"""

import os
import sys
import pandas as pd
import numpy as np
import mne
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

datadir = os.path.abspath(os.path.join(os.path.dirname( os.getcwd() ), '.', 'data'))
print(f'Data directory: {datadir}')

maindir = '' # define the main directory where the BIDS datasets are stored

# --- Global Configuration Parameters ---
task = 'rest'
phases = ['p2', 'p5']
lfreq, hfreq, fsample = 0.1, 145.0, 300.0
frange = f"{round(lfreq, 1)}-{int(hfreq)}Hz"
trans, zmm = True, 44
icselection = 'ecg04eog08'
proc = 'filt' + icselection
fitting_param = 'finley'
megtype = 'grad'
parameters = ['exponent']

# --- Paths and Subdirectories ---
# this file contains the subjects IDs and their arm number (1 or 2)
subjlistfile = os.path.join(datadir, f'meglong_{task}_subjects.tsv')
# this file contains the subjects' ages at each phase
agefile = os.path.join(datadir, f'meglong_rest_age.tsv')
deriv_folder = f'aperiodic_filt{frange}_fs{int(fsample)}Hz_trans_z{zmm}mm'
bids_project_folder = 'BIDS_long_p5_rest_arm1'
deriv_root = os.path.join(maindir, bids_project_folder, 'derivatives', deriv_folder)
statsdir = os.path.join(deriv_root, 'stats')

# --- Subject Population Data Processing ---
subjectsdf = pd.read_csv(subjlistfile, sep='\t', index_col=0)
agedf = pd.read_csv(agefile, sep='\t', index_col=0)
agedf.rename(columns={'p2_meg_age': 'p2_age', 'p5_meg_age': 'p5_age'}, inplace=True)

subjects = subjectsdf.index.tolist()
agedf = agedf.loc[subjects]
subjectsdf = subjectsdf.loc[subjects]

# Create Age Groups
age_bins = np.percentile(agedf['p2_age'], [0, 100/3, 2*100/3, 100])
age_labels = ['Young', 'Middle', 'Old']
agedf['Age_group'] = pd.cut(agedf['p2_age'], bins=age_bins, labels=age_labels, include_lowest=True)

# Build descriptive cohort boundaries dictionary
age_groups_dict = {}
for group in age_labels:
    sub_df = agedf[agedf['Age_group'] == group]
    age_groups_dict[group] = {
        'min_age': sub_df['p2_age'].min(),
        'max_age': sub_df['p2_age'].max(),
        'n_subjects': len(sub_df),
        'min_age_p5': sub_df['p5_age'].min(),
        'max_age_p5': sub_df['p5_age'].max()
    }

# --- Data Extraction Loops ---
df_vars_list = []
for knee in ['', 'knee']:
    stats_folder = f'lme_maxT_{fitting_param}{knee}_2betas_10000rand'
    varfile = os.path.join(statsdir, stats_folder, f'aperiodic_stier_{proc}_{fitting_param}{knee}_2betas_allvars_means.tsv')
    
    df_vars = pd.read_csv(varfile, sep='\t')    
    df_vars['phase'] = df_vars['subject_phase'].str.split('_').str[1]
    df_vars['subject'] = df_vars['subject_phase'].str.split('_').str[0]
    df_vars_list.append(df_vars)

age_group_colors = {'Young': 'dimgray', 'Middle': 'dodgerblue', 'Old': 'darkorange'}

# --- HELPER FUNCTIONS ---

def get_varmean(datafile, megtype):
    """Calculates channel values averaged across phases for each subject."""
    df = pd.read_csv(datafile, sep='\t').set_index(['row'])
    df = df.drop(columns=['task', 'Age0', 'deltaAge'], errors='ignore')
    
    df2 = df[df.phase == 'p2'].set_index('subject').drop(columns=['phase'], errors='ignore')
    df5 = df[df.phase == 'p5'].set_index('subject').drop(columns=['phase'], errors='ignore')
    df_mean = pd.concat([df2, df5]).groupby('subject').mean()
    
    if megtype == 'grad':
        channels = [c[:-1] + '1' for c in df_mean.columns if c.startswith('MEG') and c.endswith('2')]
        tmpdf = pd.DataFrame(index=df_mean.index)
        for c in channels:
            tmpdf[c] = df_mean[[f'{c[:-1]}2', f'{f"{c[:-1]}3"}']].mean(axis=1)
        df_mean = pd.concat([df_mean, tmpdf], axis=1)
        df_mean = df_mean.drop(columns=[c for c in df_mean.columns if c.endswith('2') or c.endswith('3')])
    else:
        channels = [c for c in df_mean.columns if c.startswith('MEG') and c.endswith('1')]

    return df_mean.mean(), df_mean.count(), channels

def get_channels_positions(channels):
    """Extracts 2D coordinate maps for topography arrays from template layout."""
    path_to_fieldtrip = '' # path to fieldtrip toolbox, which contains the layout files with the channel positions'
    layoutdir = os.path.join(path_to_fieldtrip, 'template', 'layout')
    layout = mne.channels.read_layout(os.path.join(layoutdir, 'neuromag306mag.lay'))
    
    pos = []                        
    for ch in channels:
        if ch in layout.names:
            pos.append(layout.pos[layout.names.index(ch), 0:2] / 5)
        else:
            raise ValueError(f'Channel {ch} not found in layout template file.')
    return np.array(pos)

# --- end helper functions ---

# =========================================================================
# --- Read statistics for both fixed and knee models ---
# =========================================================================
stats_df_list = []
for parameter in parameters:
    for knee in ['', 'knee']:
        stats_folder = f'lme_maxT_{fitting_param}{knee}_2betas_10000rand'
        resultsfile = os.path.join(statsdir, stats_folder, f'{parameter}_{megtype}_lme_results.tsv')
        results_df = pd.read_csv(resultsfile, sep='\t')
        print(f"\nLME results for {parameter} ({'knee' if knee else 'fixed'} model):")
        
        results_df.rename(columns={'Unnamed: 0': 'Effect', 'T-stat': 'T', 'P-val': 'P', 'Estimate': 'Beta'}, inplace=True)    

        results_df = results_df[results_df['Effect'] != '(Intercept)']
        results_df['Parameter'] = f'{parameter.replace("_", " ").title()} ({megtype})'
        results_df = results_df[['Parameter', 'Effect', 'T', 'P', 'Beta']]  
        
        results_df['P'] = np.nan  # Initialize P column with NaN values
        results_df['Kneemode'] = 'knee' if knee else 'fixed'
        

        # load original statistics
        orig_pfile = os.path.join(statsdir, stats_folder, f'aperiodic_stier_{proc}_{fitting_param}_lme_statori.npy') 
        statori = np.load(orig_pfile, allow_pickle=True).item()
        vars = statori['vars']
        vars = [v for v in vars if v.endswith(f'_{megtype}')]
        effects = statori['effects_of_interest']
        idx = vars.index(f'{parameter}_{megtype}')
      
        # load corrected p-values
        corrected_pfile = os.path.join(statsdir, stats_folder, f'aperiodic_stier_{proc}_{fitting_param}_maxT{megtype}_corrected_pvals.npy')
        corrected_pvals = np.load(corrected_pfile)

        pcorrected = corrected_pvals[idx]

        for e in effects:
            pvalue = pcorrected[effects.index(e)]
            results_df.loc[results_df['Effect'] == e, 'P'] = pvalue

        stats_df_list.append(results_df)

statsdf = pd.concat(stats_df_list, ignore_index=True)

statsdf.to_csv(os.path.join(datadir, f'supp_figure05_lme_results_combined_{megtype}.tsv'), sep='\t', index=False)

# =========================================================================
for k, kneemode in enumerate(['', 'knee']):
    df_vars = df_vars_list[k]
    knee_suffix = 'knee' if kneemode == 'knee' else 'fixed'
    
    for parameter in parameters:
        parameter_stats = statsdf[statsdf['Parameter'] == f"{parameter.replace('_', ' ').title()} ({megtype})"]
        col_y = f'{parameter}_{megtype}'

        df_data_list = []
        indiv_values = []

        for age_group in age_labels:
            group_subjects = agedf[agedf['Age_group'] == age_group].index.tolist()
            df_agegroup = df_vars[df_vars['subject'].isin(group_subjects)].copy()
            df_agegroup.sort_values(by=['subject', 'phase'], inplace=True)

            # Difference calculation for valid matched pairs to determine error bars
            tmp = df_agegroup.groupby('subject')[col_y].diff().dropna()
            sem_val = tmp.sem()                    

            for phase in phases:
                df_subset = df_vars[(df_vars['subject'].isin(group_subjects)) & (df_vars['phase'] == phase)]
                mean_value = df_subset[col_y].mean()
                indiv_values.append(df_subset[[col_y, 'subject']].set_index('subject').to_dict()[col_y])

                # Accumulate dynamically inside list object framework safely
                is_p2 = (phase == 'p2')
                df_data_list.append(pd.DataFrame({
                    'Age_group': [age_group], 'Phase': [phase], 'Mean': [mean_value], 'SEM': [sem_val],
                    'Color': [age_group_colors[age_group]], 'Alpha': [0.4 if is_p2 else 1.0],
                    'min_age': [str(age_groups_dict[age_group]['min_age'] if is_p2 else age_groups_dict[age_group]['min_age_p5'])],
                    'max_age': [str(age_groups_dict[age_group]['max_age'] if is_p2 else age_groups_dict[age_group]['max_age_p5'])],
                    'n_subjects': [str(age_groups_dict[age_group]['n_subjects'])]
                }))
        
        df_data = pd.concat(df_data_list, ignore_index=True)

        # =========================================================================
        # Save Bar Plot Summary Data (Group Means & SEMs)
        # =========================================================================
        summary_outfile = os.path.join(datadir, f'supp_figure05_barplot_summary_{knee_suffix}.tsv')
        df_data.to_csv(summary_outfile, sep='\t', index=False)
        print(f"Saved bar plot summary data to: {summary_outfile}")

        # =========================================================================
        # Save Long-Format Individual Subject Data (for Lines/Distribution)
        # =========================================================================
        df_indiv_save = df_vars[df_vars['subject'].isin(subjects)][['subject', 'phase', col_y]].copy()
        df_indiv_save = df_indiv_save.merge(agedf[['Age_group']], left_on='subject', right_index=True)
        df_indiv_save.rename(columns={col_y: parameter}, inplace=True)
        df_indiv_save['Kneemode'] = knee_suffix
        
        indiv_outfile = os.path.join(datadir, f'supp_figure05_individual_values_{knee_suffix}.tsv')
        df_indiv_save.to_csv(indiv_outfile, sep='\t', index=False)
        print(f"Saved individual subject data to: {indiv_outfile}")
        # =========================================================================
        
        # --- Topographic Map Info ---
        if 'exponent' in parameter:
            datafile = os.path.join(statsdir, f'aperiodic_stier_{proc}_{fitting_param}{kneemode}_{megtype}{parameter}_2betas.tsv')     
            varmean, _, channels = get_varmean(datafile, megtype)

            pos = get_channels_positions(channels)
            pos -= 0.1
            pos = pos * 1.2
            pos[:, 1] += 0.014 
            pos[:, 0] += 0.007

            topovals = varmean[channels].to_numpy(float)

            # =========================================================================
            # ADDITION 3: Save Spatial Channel Topography Averages
            # =========================================================================
            df_topo_save = pd.DataFrame({
                'Channel': channels,
                'Mean_Value': topovals,
                'Positions': list(pos),
                'Kneemode': knee_suffix,
                'Parameter': parameter
            })
            topo_outfile = os.path.join(datadir, f'supp_figure05_topography_means_{knee_suffix}.tsv')
            df_topo_save.to_csv(topo_outfile, sep='\t', index=False)
            print(f"Saved topography mean data to: {topo_outfile}")
            # =========================================================================


LME results for exponent (fixed model):

LME results for exponent (knee model):
Saved bar plot summary data to: /imaging/camcan/sandbox/mc06/data/supp_figure05_barplot_summary_fixed.tsv
Saved individual subject data to: /imaging/camcan/sandbox/mc06/data/supp_figure05_individual_values_fixed.tsv
Saved topography mean data to: /imaging/camcan/sandbox/mc06/data/supp_figure05_topography_means_fixed.tsv
Saved bar plot summary data to: /imaging/camcan/sandbox/mc06/data/supp_figure05_barplot_summary_knee.tsv
Saved individual subject data to: /imaging/camcan/sandbox/mc06/data/supp_figure05_individual_values_knee.tsv
Saved topography mean data to: /imaging/camcan/sandbox/mc06/data/supp_figure05_topography_means_knee.tsv


/tmp/ipykernel_1066302/1761677540.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tmpdf[c] = df_mean[[f'{c[:-1]}2', f'{f"{c[:-1]}3"}']].mean(axis=1)
/tmp/ipykernel_1066302/1761677540.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tmpdf[c] = df_mean[[f'{c[:-1]}2', f'{f"{c[:-1]}3"}']].mean(axis=1)
/tmp/ipykernel_1066302/1761677540.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u